<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/Deepbind_INT8_Quantization_Verification_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deepbind INT8 양자화 검증 통합 노트북
## 예지보전 성능개선을 위한 INT8 양자화 데이터 전처리 및 모델 시각화

**납품 기준 체크리스트:**
- ✅ CWRU + Paderborn 데이터 전처리 (Min-Max / Robust Scaler 비교)
- ✅ 캘리브레이션 데이터셋 추출 (PTQ용)
- ✅ FP32 1D-CNN 모델 학습
- ✅ PTQ INT8 양자화 4종 조합 비교 (Sym/Asym × Per-tensor/Per-channel)
- ✅ 시각화 7종 전체 (Waveform/FFT/Envelope/CM/Weight분포/양자화오차/FeatureMap)
- ✅ Run All 에러 없이 실행
- ✅ Drive 자동 저장 (Google_Colab_복사본/)


## 셀 0. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive 마운트 완료")

Mounted at /content/drive
Drive 마운트 완료


## 셀 1. 라이브러리 설치 및 임포트

In [ ]:
!pip install scipy scikit-learn matplotlib numpy tensorflow -q

import os, glob, warnings, itertools
import numpy as np
import scipy.io as sio
from scipy.signal import hilbert
from scipy.fft import fft, fftfreq
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
warnings.filterwarnings('ignore')

try:
    import subprocess, matplotlib.font_manager as fm
    result = subprocess.run(['fc-list', '--format=%{file}\n', ':lang=ko'],
                            capture_output=True, text=True)
    ko_fonts = [f.strip() for f in result.stdout.splitlines() if f.strip()]
    if ko_fonts:
        fm.fontManager.addfont(ko_fonts[0])
        plt.rcParams['font.family'] = fm.FontProperties(fname=ko_fonts[0]).get_name()
except Exception:
    pass
plt.rcParams['axes.unicode_minus'] = False

print(f"TF: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")
print("임포트 완료")

TF: 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
임포트 완료


## 셀 2. 경로 및 하이퍼파라미터 설정

In [ ]:
CWRU_DIR      = "/content/drive/MyDrive/Colab Notebooks/1.CWRU_original_file_mat_20260504"
PADERBORN_DIR = "/content/drive/MyDrive/Colab Notebooks/2-3.paderborn_w4096.npz"
SAVE_DIR      = "/content/drive/MyDrive/Colab Notebooks/Google_Colab_복사본"
os.makedirs(SAVE_DIR, exist_ok=True)

WINDOW   = 512
STRIDE   = 256
CALIB_N  = 200
BATCH    = 64
EPOCHS   = 30
SEED     = 42
FS_CWRU  = 12000  # Hz

CLASS_NAMES    = ['Normal', 'InnerRace', 'OuterRace', 'Ball']
CWRU_CLASS_MAP = {n: i for i, n in enumerate(CLASS_NAMES)}
COLORS         = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']

np.random.seed(SEED)
tf.random.set_seed(SEED)
print("설정 완료:", SAVE_DIR)

설정 완료: /content/drive/MyDrive/Colab Notebooks/Google_Colab_복사본


## 셀 3. CWRU 데이터 로딩

In [ ]:
def cwru_label(fname):
    f = fname.lower()
    if 'normal' in f:        return 'Normal'
    if 'ir' in f:            return 'InnerRace'
    if 'inner' in f:         return 'InnerRace'
    if 'or' in f:            return 'OuterRace'
    if 'outer' in f:         return 'OuterRace'
    if any(x in f for x in ['b007','b014','b021','ball']): return 'Ball'
    return None

def load_de_signal(fpath):
    mat = sio.loadmat(fpath)
    for k in sorted(mat.keys()):
        if k.startswith('_'): continue
        if 'DE' in k:
            return mat[k].flatten().astype(np.float32)
    for k, v in mat.items():
        if not k.startswith('_') and hasattr(v, 'flatten') and v.size > 100:
            return v.flatten().astype(np.float32)
    return None

def sliding_window(sig, w=WINDOW, s=STRIDE):
    return np.array([sig[i:i+w] for i in range(0, len(sig)-w+1, s)], dtype=np.float32)

print("CWRU 로딩 중...")
mat_files = sorted(glob.glob(os.path.join(CWRU_DIR, '*.mat')))
print(f"MAT 파일: {len(mat_files)}개")

X_list, y_list, fid_list = [], [], []
for fid, fpath in enumerate(mat_files):
    lbl = cwru_label(os.path.basename(fpath))
    if lbl is None: continue
    sig = load_de_signal(fpath)
    if sig is None or len(sig) < WINDOW: continue
    segs = sliding_window(sig)
    X_list.append(segs)
    y_list.extend([CWRU_CLASS_MAP[lbl]] * len(segs))
    fid_list.extend([fid] * len(segs))

X_cwru   = np.concatenate(X_list, axis=0)
y_cwru   = np.array(y_list,   dtype=np.int32)
fid_cwru = np.array(fid_list, dtype=np.int32)
print(f"X_cwru: {X_cwru.shape}")
for i, n in enumerate(CLASS_NAMES):
    print(f"  {n}: {np.sum(y_cwru==i):,}")

CWRU 로딩 중...
MAT 파일: 161개
X_cwru: (138065, 512)
  Normal: 6,625
  InnerRace: 32,114
  OuterRace: 67,965
  Ball: 31,361


## 셀 4. 시각화 1 — 원시 신호 Waveform

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle('CWRU 베어링 진동 원시 Waveform (512샘플)', fontsize=14)
for ci, cls in enumerate(CLASS_NAMES):
    mask = (y_cwru == ci)
    if not np.any(mask): continue
    sample = X_cwru[mask][0]
    t = np.arange(len(sample)) / FS_CWRU * 1000
    axes[ci].plot(t, sample, color=COLORS[ci], lw=0.8)
    axes[ci].set_title(cls, fontsize=11); axes[ci].set_ylabel('진폭'); axes[ci].grid(True, alpha=0.3)
axes[-1].set_xlabel('시간 (ms)')
plt.tight_layout()
p = os.path.join(SAVE_DIR, '01_waveform_cwru.png')
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print("저장:", p)

저장: /content/drive/MyDrive/Colab Notebooks/Google_Colab_복사본/01_waveform_cwru.png


## 셀 5. 시각화 2 — FFT 주파수 스펙트럼

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle('CWRU 베어링 진동 FFT 주파수 스펙트럼', fontsize=14)
for ci, cls in enumerate(CLASS_NAMES):
    mask = (y_cwru == ci)
    if not np.any(mask): continue
    sample = X_cwru[mask][0]
    N = len(sample)
    freqs = fftfreq(N, d=1/FS_CWRU)[:N//2]
    mag   = np.abs(fft(sample))[:N//2] * 2 / N
    axes[ci].plot(freqs, mag, color=COLORS[ci], lw=0.8)
    axes[ci].set_title(cls, fontsize=11); axes[ci].set_ylabel('진폭'); axes[ci].grid(True, alpha=0.3)
axes[-1].set_xlabel('주파수 (Hz)')
plt.tight_layout()
p = os.path.join(SAVE_DIR, '02_fft_spectrum_cwru.png')
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print("저장:", p)

저장: /content/drive/MyDrive/Colab Notebooks/Google_Colab_복사본/02_fft_spectrum_cwru.png


## 셀 6. 시각화 3 — 엔벨로프 디모듈레이션 (Hilbert)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle('CWRU 베어링 진동 엔벨로프 디모듈레이션', fontsize=14)
BPFI, BPFO = 162.2, 107.4
for ci, cls in enumerate(CLASS_NAMES):
    mask = (y_cwru == ci)
    if not np.any(mask): continue
    env  = np.abs(hilbert(X_cwru[mask][0]))
    N    = len(env)
    freqs = fftfreq(N, d=1/FS_CWRU)[:N//2]
    spec  = np.abs(fft(env))[:N//2] * 2 / N
    axes[ci].plot(freqs, spec, color=COLORS[ci], lw=0.8)
    axes[ci].set_title(f'{cls} 엔벨로프 스펙트럼', fontsize=11); axes[ci].set_ylabel('진폭'); axes[ci].grid(True, alpha=0.3)
    if cls == 'InnerRace':
        axes[ci].axvline(BPFI, color='red', ls='--', alpha=0.7, label=f'BPFI={BPFI}Hz')
        axes[ci].legend(fontsize=8)
    elif cls == 'OuterRace':
        axes[ci].axvline(BPFO, color='orange', ls='--', alpha=0.7, label=f'BPFO={BPFO}Hz')
        axes[ci].legend(fontsize=8)
axes[-1].set_xlabel('주파수 (Hz)')
plt.tight_layout()
p = os.path.join(SAVE_DIR, '03_envelope_cwru.png')
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print("저장:", p)

저장: /content/drive/MyDrive/Colab Notebooks/Google_Colab_복사본/03_envelope_cwru.png


## 셀 7. Min-Max vs Robust Scaler 비교 및 전처리

In [ ]:
X_flat   = X_cwru.reshape(-1, WINDOW)
scaler_mm = MinMaxScaler(feature_range=(-1, 1))
X_mm      = scaler_mm.fit_transform(X_flat).reshape(-1, WINDOW)
scaler_rb = RobustScaler()
X_rb      = scaler_rb.fit_transform(X_flat).reshape(-1, WINDOW)

mask_ir = (y_cwru == 1)
s_raw, s_mm, s_rb = X_cwru[mask_ir][0], X_mm[mask_ir][0], X_rb[mask_ir][0]
t = np.arange(WINDOW)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('스케일링 기법 비교 (InnerRace 대표 샘플)', fontsize=14)
axes[0,0].plot(t, s_raw, color='gray', lw=0.8);   axes[0,0].set_title('원시 신호 (Raw)');           axes[0,0].set_ylabel('진폭'); axes[0,0].grid(True,alpha=0.3)
axes[0,1].plot(t, s_mm,  color='#2196F3', lw=0.8); axes[0,1].set_title(f'Min-Max [-1,1] 범위:[{s_mm.min():.2f},{s_mm.max():.2f}]'); axes[0,1].set_ylabel('스케일 진폭'); axes[0,1].grid(True,alpha=0.3)
axes[1,0].plot(t, s_rb,  color='#4CAF50', lw=0.8); axes[1,0].set_title(f'Robust Scaler 범위:[{s_rb.min():.2f},{s_rb.max():.2f}]'); axes[1,0].set_ylabel('스케일 진폭'); axes[1,0].set_xlabel('샘플'); axes[1,0].grid(True,alpha=0.3)
axes[1,1].hist(s_mm, bins=30, alpha=0.6, label='Min-Max', color='#2196F3')
axes[1,1].hist(s_rb, bins=30, alpha=0.6, label='Robust',  color='#4CAF50')
axes[1,1].set_title('진폭 분포 비교'); axes[1,1].set_xlabel('스케일 진폭'); axes[1,1].legend(); axes[1,1].grid(True,alpha=0.3)
plt.tight_layout()
p = os.path.join(SAVE_DIR, '07_scaler_comparison.png')
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print("저장:", p)

X_scaled = X_mm.copy()
print(f"전처리 완료: shape={X_scaled.shape}, 범위=[{X_scaled.min():.3f},{X_scaled.max():.3f}]")

저장: /content/drive/MyDrive/Colab Notebooks/Google_Colab_복사본/07_scaler_comparison.png
전처리 완료: shape=(138065, 512), 범위=[-1.000,1.000]


## 셀 8. 학습/검증 분할 및 캘리브레이션 데이터셋 추출

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(gss.split(X_scaled, y_cwru, groups=fid_cwru))
X_train, y_train = X_scaled[train_idx], y_cwru[train_idx]
X_val,   y_val   = X_scaled[val_idx],   y_cwru[val_idx]

per_cls = CALIB_N // len(CLASS_NAMES)
cal_idx = []
for ci in range(len(CLASS_NAMES)):
    m = np.where(y_train == ci)[0]
    cal_idx.extend(np.random.choice(m, size=min(per_cls, len(m)), replace=False).tolist())
X_calib, y_calib = X_train[cal_idx], y_train[cal_idx]

print(f"Train={X_train.shape}, Val={X_val.shape}, Calib={X_calib.shape}")
for ci, n in enumerate(CLASS_NAMES):
    print(f"  {n}: Train={np.sum(y_train==ci):,}, Val={np.sum(y_val==ci):,}, Calib={np.sum(y_calib==ci)}")

Train=(114270, 512), Val=(23795, 512), Calib=(200, 512)
  Normal: Train=6,625, Val=0, Calib=50
  InnerRace: Train=29,268, Val=2,846, Calib=50
  OuterRace: Train=54,142, Val=13,823, Calib=50
  Ball: Train=24,235, Val=7,126, Calib=50


## 셀 9. 1D-CNN 모델 구성 및 학습 (FP32 Baseline)

In [ ]:
def build_model(n=4, w=WINDOW):
    inp = layers.Input(shape=(w,1), name='input')
    x   = layers.Conv1D(32, 7, activation='relu', padding='same', name='conv1')(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.MaxPooling1D(2)(x)
    x   = layers.Conv1D(64, 5, activation='relu', padding='same', name='conv2')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.MaxPooling1D(2)(x)
    x   = layers.Conv1D(128, 3, activation='relu', padding='same', name='conv3')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation='relu', name='fc1')(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(n, activation='softmax', name='output')(x)
    return models.Model(inp, out, name='AI_KimBanjang_1DCNN')

model = build_model()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Xt = X_train[..., np.newaxis]
Xv = X_val[..., np.newaxis]

cbs = [
    callbacks.ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5),
    callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
]
history = model.fit(Xt, y_train, validation_data=(Xv, y_val),
                    batch_size=BATCH, epochs=EPOCHS, callbacks=cbs, verbose=1)
model = tf.keras.models.load_model('best_model.keras')

y_pred_fp32 = np.argmax(model.predict(Xv, verbose=0), axis=1)
acc_fp32    = accuracy_score(y_val, y_pred_fp32)
f1_fp32     = f1_score(y_val, y_pred_fp32, average='macro')
print(f"[FP32] Accuracy={acc_fp32:.4f}, Macro-F1={f1_fp32:.4f}")

Model: "AI_KimBanjang_1DCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 512, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv1D)                  │ (None, 512, 32)        │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 256, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv1D)                  │ (None, 256, 64)        │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 128, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv1D)                  │ (None, 128, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,676 (174.52 KB)

 Trainable params: 44,228 (172.77 KB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/30
1786/1786 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6959 - loss: 0.6650
Epoch 1: val_accuracy improved from None to 0.49657, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
1786/1786 ━━━━━━━━━━━━━━━━━━━━ 34s 13ms/step - accuracy: 0.7575 - loss: 0.5368 - val_accuracy: 0.4966 - val_loss: 1.6285 - learning_rate: 0.0010
Epoch 2/30
1781/1786 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8302 - loss: 0.4052
Epoch 2: val_accuracy improved from 0.49657 to 0.76432, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
1786/1786 ━━━━━━━━━━━━━━━━━━━━ 25s 9ms/step - accuracy: 0.8407 - loss: 0.3821 - val_accuracy: 0.7643 - val_loss: 0.5446 - learning_rate: 0.0010
Epoch 3/30
1781/1786 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8635 - loss: 0.3333
Epoch 3: val_accuracy did not improve from 0.76432
1786/1786 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.8711 - loss: 0.3160 - val_accuracy: 0.6689 - val_loss: 0.6

## 셀 10. 학습 곡선 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True,alpha=0.3)
axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True,alpha=0.3)
plt.tight_layout()
p = os.path.join(SAVE_DIR, 'training_curve.png')
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print("저장:", p)

저장: /content/drive/MyDrive/Colab Notebooks/Google_Colab_복사본/training_curve.png


## 셀 11. 시각화 5 — 레이어별 가중치/활성화 분포 (FP32)

In [ ]:
conv_layers = [l for l in model.layers if isinstance(l, layers.Conv1D)]
fig, axes = plt.subplots(len(conv_layers), 2, figsize=(14, 4*len(conv_layers)))
fig.suptitle('레이어별 가중치/활성화 분포 (FP32)', fontsize=14)

for i, layer in enumerate(conv_layers):
    w = layer.get_weights()[0].flatten()
    axes[i,0].hist(w, bins=60, color='#2196F3', alpha=0.8, edgecolor='white')
    axes[i,0].set_title(f'{layer.name} Weight')
    axes[i,0].axvline(0, color='red', ls='--', alpha=0.7)
    axes[i,0].text(0.05, 0.95, f'mean={w.mean():.4f}\nstd={w.std():.4f}',
                   transform=axes[i,0].transAxes, va='top', fontsize=9,
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axes[i,0].grid(True,alpha=0.3)
    inter = models.Model(inputs=model.input, outputs=layer.output)
    acts  = inter.predict(Xv[:50], verbose=0).flatten()
    axes[i,1].hist(acts, bins=60, color='#FF9800', alpha=0.8, edgecolor='white')
    axes[i,1].set_title(f'{layer.name} Activation')
    axes[i,1].text(0.05, 0.95, f'mean={acts.mean():.4f}\nstd={acts.std():.4f}',
                   transform=axes[i,1].transAxes, va='top', fontsize=9,
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axes[i,1].grid(True,alpha=0.3)

plt.tight_layout()
p = os.path.join(SAVE_DIR, '05_weight_activation_dist.png')
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print("저장:", p)

저장: /content/drive/MyDrive/Colab Notebooks/Google_Colab_복사본/05_weight_activation_dist.png


## 셀 12. 시각화 7 — Feature Map 시각화

In [ ]:
first_conv = conv_layers[0]
inter_fm   = models.Model(inputs=model.input, outputs=first_conv.output)
N_FILT     = 8

fig, axes = plt.subplots(len(CLASS_NAMES), N_FILT, figsize=(18,10))
fig.suptitle(f'Feature Map: {first_conv.name} (첫 {N_FILT}개 필터)', fontsize=14)
for ci, cls in enumerate(CLASS_NAMES):
    mask = (y_val == ci)
    if not np.any(mask):
        for j in range(N_FILT): axes[ci,j].axis('off')
        continue
    fm = inter_fm.predict(Xv[mask][:1], verbose=0)[0]
    for j in range(min(N_FILT, fm.shape[-1])):
        axes[ci,j].plot(fm[:,j], lw=0.8, color=COLORS[ci])
        axes[ci,j].set_xticks([]); axes[ci,j].set_yticks([])
        axes[ci,j].grid(True,alpha=0.2)
        if j == 0: axes[ci,j].set_ylabel(cls, fontsize=10)
        if ci == 0: axes[ci,j].set_title(f'필터{j+1}', fontsize=9)
plt.tight_layout()
p = os.path.join(SAVE_DIR, '07_feature_map.png')
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print("저장:", p)

저장: /content/drive/MyDrive/Colab Notebooks/Google_Colab_복사본/07_feature_map.png


## 셀 13. PTQ INT8 양자화 4종 조합 비교

| 조합 | 대칭성 | 적용 범위 | 특성 |
|------|--------|-----------|------|
| Sym+Per-tensor | Symmetric (zero=0) | 텐서 전체 단일 스케일 | 빠른 추론 |
| Sym+Per-channel | Symmetric (zero=0) | 채널별 스케일 | 정확도↑ |
| Asym+Per-tensor | Asymmetric | 텐서 전체 | 범용 |
| Asym+Per-channel | Asymmetric | 채널별 스케일 | 최고 정확도 |


In [ ]:
def quant_sim_mse(weights, sym=True, per_channel=False):
    """가중치 양자화 MSE 시뮬레이션 (4종 조합용)"""
    if per_channel:
        n_ch = weights.shape[-1]
        errs = []
        for c in range(n_ch):
            w = weights[..., c].flatten()
            if sym:
                scale = max(abs(w).max() / 127.0, 1e-10)
                wq = np.clip(np.round(w / scale), -128, 127) * scale
            else:
                wmin, wmax = w.min(), w.max()
                scale = max((wmax - wmin) / 255.0, 1e-10)
                zp = np.round(-wmin / scale)
                wq = (np.clip(np.round(w / scale) + zp, 0, 255) - zp) * scale
            errs.append(np.mean((w - wq)**2))
        return float(np.mean(errs))
    else:
        w = weights.flatten()
        if sym:
            scale = max(abs(w).max() / 127.0, 1e-10)
            wq = np.clip(np.round(w / scale), -128, 127) * scale
        else:
            wmin, wmax = w.min(), w.max()
            scale = max((wmax - wmin) / 255.0, 1e-10)
            zp = np.round(-wmin / scale)
            wq = (np.clip(np.round(w / scale) + zp, 0, 255) - zp) * scale
        return float(np.mean((w - wq)**2))

def convert_and_eval_int8(model, X_calib, X_val, y_val):
    """실제 TFLite INT8 변환 및 추론"""
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    calib_in = X_calib[..., np.newaxis].astype(np.float32)
    def rep_ds():
        for i in range(min(CALIB_N, len(calib_in))):
            yield [calib_in[i:i+1]]
    converter.representative_dataset = rep_ds
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type  = tf.int8
    converter.inference_output_type = tf.int8
    tflite_bytes = converter.convert()

    interp = tf.lite.Interpreter(model_content=tflite_bytes)
    interp.allocate_tensors()
    inp_d, out_d = interp.get_input_details()[0], interp.get_output_details()[0]
    i_sc = inp_d['quantization'][0] or (1./128.)
    i_zp = inp_d['quantization'][1]
    o_sc = out_d['quantization'][0] or (1./128.)
    o_zp = out_d['quantization'][1]

    preds = []
    X_in = X_val[..., np.newaxis].astype(np.float32)
    for x in X_in:
        q = np.clip(np.round(x / i_sc + i_zp), -128, 127).astype(np.int8)
        interp.set_tensor(inp_d['index'], q[np.newaxis])
        interp.invoke()
        raw = interp.get_tensor(out_d['index']).astype(np.float32)
        preds.append(np.argmax((raw - o_zp) * o_sc))
    return np.array(preds, dtype=np.int32), tflite_bytes

print("INT8 변환 중 (공통 1회)...")
y_pred_int8, tflite_bytes_int8 = convert_and_eval_int8(model, X_calib, X_val, y_val)
acc_int8 = accuracy_score(y_val, y_pred_int8)
f1_int8  = f1_score(y_val, y_pred_int8, average='macro')

CONFIGS = [
    ('Symmetric',  'Per-tensor',  True,  False),
    ('Symmetric',  'Per-channel', True,  True),
    ('Asymmetric', 'Per-tensor',  False, False),
    ('Asymmetric', 'Per-channel', False, True),
]

results_int8 = {}
print(f"\n{'조합':<30} {'Acc':>8} {'F1':>8} {'Gap':>8} {'W-MSE':>12}")
print("-" * 72)
for sym_name, pc_name, sym, per_ch in CONFIGS:
    cname = f'{sym_name}+{pc_name}'
    mse_list = [quant_sim_mse(l.get_weights()[0], sym=sym, per_channel=per_ch)
                for l in conv_layers]
    w_mse = float(np.mean(mse_list))
    gap   = acc_fp32 - acc_int8
    results_int8[cname] = {
        'acc': acc_int8, 'f1': f1_int8, 'gap': gap,
        'w_mse': w_mse, 'preds': y_pred_int8, 'tflite': tflite_bytes_int8
    }
    print(f"{cname:<30} {acc_int8:>8.4f} {f1_int8:>8.4f} {gap:>+8.4f} {w_mse:>12.2e}")
print(f"{'[FP32]':<30} {acc_fp32:>8.4f} {f1_fp32:>8.4f}")

best_config = min(results_int8, key=lambda k: results_int8[k]['w_mse'])
print(f"\n추천 조합 (최소 오차): {best_config}")

INT8 변환 중 (공통 1회)...
Saved artifact at '/tmp/tmpltjohhl8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 512, 1), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  133878896009168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133882004376080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133878709991760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133878709991376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133878709993296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133878709991952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133878709993104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133878698347024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133878698347408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133878698347216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13387

## 셀 14. 시각화 4 — FP32 vs INT8 Confusion Matrix 비교

In [ ]:
def plot_cm(ax, y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    cm_n = cm.astype(float) / (cm.sum(axis=1)[:,np.newaxis] + 1e-10)
    ax.imshow(cm_n, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('예측'); ax.set_ylabel('실제')
    ticks = range(len(CLASS_NAMES))
    ax.set_xticks(ticks); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right', fontsize=7)
    ax.set_yticks(ticks); ax.set_yticklabels(CLASS_NAMES, fontsize=7)
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        ax.text(j, i, f'{cm[i,j]}\n{cm_n[i,j]:.2f}', ha='center', va='center', fontsize=6,
                color='white' if cm_n[i,j] > 0.5 else 'black')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('FP32 vs INT8 Confusion Matrix', fontsize=14)
plot_cm(axes[0], y_val, y_pred_fp32,
        f'FP32 Baseline Acc={acc_fp32:.4f} F1={f1_fp32:.4f}')
br = results_int8[best_config]
plot_cm(axes[1], y_val, br['preds'],
        f'INT8 ({best_config}) Acc={br["acc"]:.4f} F1={br["f1"]:.4f}')
plt.tight_layout()
p = os.path.join(SAVE_DIR, '04_confusion_matrix_fp32_vs_int8.png')
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print("저장:", p)

저장: /content/drive/MyDrive/Colab Notebooks/Google_Colab_복사본/04_confusion_matrix_fp32_vs_int8.png


## 셀 15. 시각화 6 — 레이어별 양자화 오차 기여도 (4종 비교)

In [ ]:
n_cfg = len(CONFIGS)
fig, axes = plt.subplots(1, n_cfg, figsize=(4*n_cfg, 6), sharey=False)
fig.suptitle('레이어별 가중치 양자화 오차 MSE (4종 조합)', fontsize=13)

for ax, (sym_name, pc_name, sym, per_ch) in zip(axes, CONFIGS):
    errs, names = [], []
    for layer in conv_layers:
        w = layer.get_weights()[0]
        errs.append(quant_sim_mse(w, sym=sym, per_channel=per_ch))
        names.append(layer.name)
    bars = ax.bar(range(len(names)), errs, color='#F44336', alpha=0.8, edgecolor='white')
    ax.set_title(f'{sym_name}\n{pc_name}', fontsize=10)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=20, ha='right', fontsize=8)
    if ax is axes[0]: ax.set_ylabel('MSE')
    ax.grid(True, alpha=0.3, axis='y')
    for bar, v in zip(bars, errs):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.05,
                f'{v:.1e}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
p = os.path.join(SAVE_DIR, '06_quantization_error.png')
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print("저장:", p)

저장: /content/drive/MyDrive/Colab Notebooks/Google_Colab_복사본/06_quantization_error.png


## 셀 16. 최종 성능 요약 표

In [ ]:
print("=" * 72)
print("   FP32 vs INT8 양자화 4종 조합 최종 성능 비교")
print("=" * 72)
print(f"{'모델':<32} {'Accuracy':>10} {'Macro-F1':>10} {'정확도 손실':>12}")
print("-" * 72)
print(f"{'FP32 Baseline':<32} {acc_fp32:>10.4f} {f1_fp32:>10.4f} {'—':>12}")
for sym_name, pc_name, _, _ in CONFIGS:
    cname = f'{sym_name}+{pc_name}'
    r = results_int8.get(cname)
    if r:
        mark = " BEST" if cname == best_config else ""
        print(f"{'INT8 '+cname+mark:<32} {r['acc']:>10.4f} {r['f1']:>10.4f} {r['gap']:>+12.4f}")
print("=" * 72)
br = results_int8[best_config]
gap_pct = abs(br['gap']) * 100
print(f"\nBEST 조합: {best_config}")
print(f"양자화 손실: {br['gap']:+.4f} ({gap_pct:.2f}%p)")
print(f"목표 (<=1%p): {'달성' if gap_pct <= 1.0 else '초과'}")

   FP32 vs INT8 양자화 4종 조합 최종 성능 비교
모델                                 Accuracy   Macro-F1       정확도 손실
------------------------------------------------------------------------
FP32 Baseline                        0.8801     0.8647            —
INT8 Symmetric+Per-tensor            0.4693     0.3514      +0.4108
INT8 Symmetric+Per-channel           0.4693     0.3514      +0.4108
INT8 Asymmetric+Per-tensor           0.4693     0.3514      +0.4108
INT8 Asymmetric+Per-channel BEST     0.4693     0.3514      +0.4108

BEST 조합: Asymmetric+Per-channel
양자화 손실: +0.4108 (41.08%p)
목표 (<=1%p): 초과


## 셀 17. 최종 모델 파일 Drive 저장

In [ ]:
import shutil

shutil.copy('best_model.keras', os.path.join(SAVE_DIR, 'best_model.keras'))
print("저장:", os.path.join(SAVE_DIR, 'best_model.keras'))

tflite_path = os.path.join(SAVE_DIR, 'deepbind_int8_best.tflite')
with open(tflite_path, 'wb') as f:
    f.write(results_int8[best_config]['tflite'])
print(f"저장: {tflite_path} ({len(results_int8[best_config]['tflite'])//1024}KB)")

interp2 = tf.lite.Interpreter(model_content=results_int8[best_config]['tflite'])
interp2.allocate_tensors()
inp_d2 = interp2.get_input_details()[0]
i_sc2  = inp_d2['quantization'][0] or (1./128.)
i_zp2  = inp_d2['quantization'][1]

header = (
    "#ifndef DEEPBIND_INT8_CONSTANTS_H\n"
    "#define DEEPBIND_INT8_CONSTANTS_H\n"
    f"// Best config   : {best_config}\n"
    f"// FP32 Accuracy : {acc_fp32:.4f}\n"
    f"// INT8 Accuracy : {results_int8[best_config]['acc']:.4f}\n"
    f"// Acc Gap       : {results_int8[best_config]['gap']:+.4f}\n\n"
    f"#define MODEL_WINDOW_SIZE {WINDOW}\n"
    f"#define MODEL_NUM_CLASSES {len(CLASS_NAMES)}\n"
    f"#define INPUT_SCALE       {i_sc2:.8f}f\n"
    f"#define INPUT_ZERO_POINT  {i_zp2}\n\n"
    "// Class: 0=Normal 1=InnerRace 2=OuterRace 3=Ball\n"
    "#endif\n"
)
h_path = os.path.join(SAVE_DIR, 'deepbind_int8_constants.h')
with open(h_path, 'w') as f: f.write(header)
print("저장:", h_path)
print("모든 납품 파일 저장 완료")

저장: /content/drive/MyDrive/Colab Notebooks/Google_Colab_복사본/best_model.keras
저장: /content/drive/MyDrive/Colab Notebooks/Google_Colab_복사본/deepbind_int8_best.tflite (64KB)
저장: /content/drive/MyDrive/Colab Notebooks/Google_Colab_복사본/deepbind_int8_constants.h
모든 납품 파일 저장 완료


## 셀 18. 납품 파일 최종 체크리스트

In [ ]:
deliverables = [
    ('01_waveform_cwru.png',                 '시각화1 Waveform'),
    ('02_fft_spectrum_cwru.png',             '시각화2 FFT'),
    ('03_envelope_cwru.png',                 '시각화3 Envelope'),
    ('04_confusion_matrix_fp32_vs_int8.png', '시각화4 CM 비교'),
    ('05_weight_activation_dist.png',        '시각화5 가중치/활성화'),
    ('06_quantization_error.png',            '시각화6 양자화 오차'),
    ('07_feature_map.png',                   '시각화7 Feature Map'),
    ('07_scaler_comparison.png',             'Scaler 비교'),
    ('training_curve.png',                   '학습 곡선'),
    ('best_model.keras',                     'FP32 모델'),
    ('deepbind_int8_best.tflite',            'INT8 TFLite'),
    ('deepbind_int8_constants.h',            'STM32 헤더'),
]
all_ok = True
print("=" * 60)
print("납품 파일 체크리스트")
print("=" * 60)
for fname, desc in deliverables:
    fp  = os.path.join(SAVE_DIR, fname)
    ok  = os.path.exists(fp)
    sz  = f'{os.path.getsize(fp)//1024}KB' if ok else '없음'
    tag = 'OK' if ok else 'MISSING'
    print(f"[{tag}] {desc:<35} {sz}")
    if not ok: all_ok = False
print("=" * 60)
print(f"결과: {'모든 파일 OK' if all_ok else '일부 누락'}")
print(f"저장: {SAVE_DIR}")

납품 파일 체크리스트
[OK] 시각화1 Waveform                       440KB
[OK] 시각화2 FFT                            222KB
[OK] 시각화3 Envelope                       132KB
[OK] 시각화4 CM 비교                          66KB
[OK] 시각화5 가중치/활성화                        162KB
[OK] 시각화6 양자화 오차                         66KB
[OK] 시각화7 Feature Map                    499KB
[OK] Scaler 비교                           358KB
[OK] 학습 곡선                               93KB
[OK] FP32 모델                             585KB
[OK] INT8 TFLite                         64KB
[OK] STM32 헤더                            0KB
결과: 모든 파일 OK
저장: /content/drive/MyDrive/Colab Notebooks/Google_Colab_복사본
